# Validate converted corpus files
* Install & set up dependencies
* extract schematron from RNG and transform to XSLT
* run schematron-xslt on files

In [ ]:
import os
import requests
import pandas as pd
from pathlib import Path
import saxonche
from zipfile import ZipFile
from lxml import etree

In [ ]:
SCRIPT_DIR = Path(__file__).resolve().parent
tmpDir = str(SCRIPT_DIR / "tmp")
libDir = str(SCRIPT_DIR / "lib")
os.makedirs(tmpDir, exist_ok=True)
os.makedirs(libDir, exist_ok=True)
os.makedirs(os.path.join(tmpDir, "validationReports"), exist_ok=True)
namespaces = {"tei": "http://www.tei-c.org/ns/1.0"}
# the root of the git repository
dataHome = "../.."

# rng schema
rngSchema = dataHome + "/803_RNG_Schematron/wibarab_corpus.rng"
corpusDir = dataHome + "/103_tei_w"

errorReport = os.path.join(tmpDir, "validation_report.html")

with saxonche.PySaxonProcessor(license=False) as proc:
    print(proc.version)
    proc.set_cwd(os.path.dirname(os.path.dirname(os.path.abspath(""))))
    print(proc.cwd)

In [ ]:


def downloadAndStore(url, force=False):
    #  filename of the file to be downloaded
    fn = os.path.basename(url)
    # fn w/o extension
    basename = os.path.splitext(fn)[0]
    # extension
    ext = os.path.splitext(fn)[1]
    dlFilePath = tmpDir + "/" + fn
    if force or not os.path.exists(dlFilePath):
        payload = requests.get(url).content
        open(dlFilePath, "wb").write(payload)
    return dlFilePath

In [ ]:


def downloadAndUnzip(url):
    #  filename of the file to be downloaded
    fn = os.path.basename(url)
    # fn w/o extension
    basename = os.path.splitext(fn)[0]
    # extension
    ext = os.path.splitext(fn)[1]

    if ext != ".zip":
        return "not a zip archive"
    else:
        zipFilePath = downloadAndStore(url)
        # the path where the content should be extracted to
        targetPath = libDir + "/" + basename

        payload = requests.get(url).content
        open(zipFilePath, "wb").write(payload)
        ZipFile(zipFilePath).extractall(path=targetPath)

    return targetPath

### Install XSL based schematron validator

In [ ]:
_schCompiler = None


def setupSchXSLT():
    global _schCompiler
    if _schCompiler is not None:
        return _schCompiler
    schDLURL = "https://codeberg.org/SchXslt/schxslt/releases/download/v1.10.1/schxslt-1.10.1-xslt-only.zip"
    schHome = downloadAndUnzip(schDLURL)
    _schCompiler = schHome + "/schxslt-1.10.1/2.0/pipeline-for-svrl.xsl"
    if os.path.exists(_schCompiler):
        return _schCompiler
    else:
        print("error: something went wrong, cannot locate file '" + _schCompiler + "'")

In [ ]:
setupSchXSLT()

In [ ]:


def transform(s, xsl, o, parameters=None):
    # processor keeps files open on Windows and in doing so prevents moving or copying them
    try:
        with saxonche.PySaxonProcessor(license=False) as proc:
            proc.set_configuration_property("xi", "on")
            saxon = proc.new_xslt30_processor()
            if parameters is None:
                parameters = {}
            for i in parameters:
                saxon.set_parameter(name=i, value=proc.make_string_value(parameters[i]))
            exec = saxon.compile_stylesheet(stylesheet_file=os.path.abspath(xsl))
            exec.apply_templates_returning_file(
                source_file=os.path.abspath(s), output_file=os.path.abspath(o)
            )
            if exec.exception_occurred:
                exec.get_error_message
                # for i in range(saxon.exception_count()-1):
                print(saxon.get_error_message())
                print(
                    os.path.abspath(s)
                    + " - "
                    + os.path.abspath(xsl)
                    + " -> "
                    + os.path.abspath(o)
                    + " failed"
                )
            if os.path.exists(os.path.abspath(o)):
                return o
            else:
                print(
                    "there was an error transforming " + s + " with stylesheet " + xsl
                )
    except Exception as e:
        print("Python Exception during transform:", e)

## Prepare rng2sch stylesheet

Returns path to the xsl that extracts schematron form the RelaxNG schema.
This should only run once as the file gets locked (by saxon) and so further attempts to pring it to the correct location will fail.

In [ ]:


_rng2sch = None

def setupRNG2Sch():
    global _rng2sch
    if _rng2sch is not None:
        return _rng2sch
    RNG2SchtrDL = "https://raw.githubusercontent.com/Schematron/schematron/master/trunk/converters/code/ToSchematron/ExtractSchFromRNG.xsl"
    dltmp = downloadAndStore(RNG2SchtrDL)
    # tweak XSLT
    with open(dltmp, encoding="utf-8") as inputfile:
        lines = inputfile.read()
    lines = lines.replace(
        "http://www.ascc.net/xml/schematron", "http://purl.oclc.org/dsdl/schematron/"
    )
    lines = lines.replace("<sch:schema", '<sch:schema queryBinding="xslt2"')

    with open(dltmp, "w", encoding="utf-8") as file:
        file.writelines(lines)
    _rng2sch = libDir + "/" + os.path.basename(dltmp)
    os.replace(dltmp, _rng2sch)
    if os.path.exists(_rng2sch):
        return _rng2sch
    else:
        print("error: something went wrong, cannot locate file '" + _rng2sch + "'")

In [ ]:
setupRNG2Sch()

## Extract schematron from RNG and transform to XSLT

In [ ]:


def extractSchematron(rng):
    """extracts a schematron document embedded in an rng schema"""
    print("extracting Schematron document from " + rng)
    rng2sch = setupRNG2Sch()
    sch = tmpDir + "/" + os.path.basename(rng) + ".sch"
    if not os.path.exists(sch):
        transform(rng, rng2sch, sch)
    return sch

In [ ]:
def compileSchematron(sch):
    """compiles a schematron document to an XSLT stylesheet"""
    outputPath = tmpDir + "/" + os.path.basename(sch) + ".xsl"
    schCompiler = setupSchXSLT()
    transform(sch, schCompiler, outputPath)

    if os.path.exists(outputPath):
        print(outputPath)
        return outputPath
    else:
        print("error: something went wrong, cannot locate file '" + outputPath + "'")

In [ ]:
sch = extractSchematron(rngSchema)
schXSL = compileSchematron(sch)

## Run schematron and relaxNG on files

In [ ]:
validationErrors = []
ignoredFiles = []

In [ ]:
def schValidate(sch, path):
    """Validates a document (at path) against a Schematron schema (at sch). Returns a list of dicts for errors and successful checks."""
    results = []
    out = tmpDir + "/validationReports/" + os.path.basename(path)
    xsl = compileSchematron(sch)
    try:
        transform(path, xsl, out)
    except saxonche.PySaxonApiError as e:
        print(f"An error occurred while running schValidate on {path}")
        return []

    report = etree.parse(out)
    source_doc = etree.parse(path)
    failedAssert = report.findall("{http://purl.oclc.org/dsdl/svrl}failed-assert")
    successfulReport = report.findall(
        "{http://purl.oclc.org/dsdl/svrl}successful-report"
    )

    for s in successfulReport + failedAssert:
        xpath = (
            s.attrib["location"]
            .replace("Q{http://www.tei-c.org/ns/1.0}", "tei:")
            .replace("Q{}", "")
        )
        msg = s.find("{http://purl.oclc.org/dsdl/svrl}text").text

        resolved_line = None
        if xpath:
            try:
                matches = source_doc.xpath(xpath, namespaces=namespaces)
                if matches:
                    first = matches[0]
                    if isinstance(first, etree._Element):
                        resolved_line = first.sourceline
                    elif hasattr(first, "getparent") and first.getparent() is not None:
                        resolved_line = first.getparent().sourceline
            except Exception:
                resolved_line = None

        results.append(
            {
                "type": "error",
                "message": msg,
                "line": resolved_line,
                "location": xpath,
                "source": path,
                "stage": "schematron",
                "exceptionType": str(s.tag).replace(
                    "{http://purl.oclc.org/dsdl/svrl}", ""
                ),
            }
        )
    return results

In [ ]:
def validate(path, rngSchema):
    """Validate a document against the rngSchema. Returns a list of dicts of which each one represents a validation (or parsing) error."""
    validationErrors = []
    sch = extractSchematron(rngSchema)
    try:
        doc = etree.parse(path)
        # relaxng validation
        relaxng_doc = etree.parse(rngSchema)
        relaxng = etree.RelaxNG(relaxng_doc)
        relaxng.assertValid(doc)
        # schematron validation
        schErrs = schValidate(sch, path)
        if len(schErrs) >= 1:
            validationErrors = validationErrors + schErrs

    except etree.XMLSyntaxError as e:
        valErrObj = {
            "type": "error",
            "message": str(e),
            "line": e.lineno,
            "source": path,
            "location": "n/a",
            "stage": "parsing",
            "exceptionType": type(e).__name__,
        }
        return valErrObj
    except etree.RelaxNGValidateError as e:
        for error in e.error_log:
            print("relaxNG error: ", error)
    except etree.DocumentInvalid as e:
        for error in e.error_log:
            # print(error)
            # we ignore rng errors about @schemaLocation since
            # that is needed for validation in the TEI-enricher
            if error.message != "Invalid attribute schemaLocation for element TEI":
                location = "n/a" if error.path is None else error.path
                valErrObj = {
                    "type": "error",
                    "message": error.message,
                    "line": error.line,
                    "source": path,
                    "location": location,
                    "stage": "relaxng",
                    "exceptionType": type(e).__name__,
                }
                # DEBUG
                print(valErrObj)

                validationErrors.append(valErrObj)

        # if the document is invalid against the RNG, we still want to run schematron against it
        schErrs = schValidate(sch, path)
        if len(schErrs) >= 1:
            validationErrors = validationErrors + schErrs

    return validationErrors

In [ ]:
def docStatus(path):
    """Checks XML well-formedness. Returns True when parseable, else a dict with parse error details."""
    try:
        etree.parse(path)
        return True
    # report documents which are not well-formed
    except etree.XMLSyntaxError as e:
        return {
            "type": "error",
            "message": str(e),
            "line": e.lineno,
            "source": path,
            "location": "n/a",
            "stage": "parsing",
            "exceptionType": type(e).__name__,
        }

In [ ]:
def validateAndAppend(filepath, rngSchema):
    global validationErrors, ignoredFiles
    print("validating " + filepath)
    results = validate(filepath, rngSchema)
    # print(results)
    len(results)
    if type(results) is list:
        res_errs = filter(lambda x: x["type"] == "error", results)
        current_errs = list(res_errs)
        validationErrors = validationErrors + current_errs
        print(f"{len(current_errs)} found / {len(validationErrors)} in total")
    elif type(results) is dict:
        if results["type"] == "ignored":
            ignoredFiles.append(results)
    else:
        print("unknown result type")
        print(results)

In [ ]:
def validate_corpus_file(i, folder):
    filename = os.path.basename(i)
    filepath = folder + "/" + filename
    status = docStatus(filepath)
    print(filename, status)

    # if the document couldn't be parsed, docStatus() returns a dict
    # with some error information which is appended to the list of
    # validation errors
    if type(status) is dict and status["type"] == "error":
        validationErrors.append(status)

    # … otherwise try to validate the document
    else:
        validateAndAppend(filepath, rngSchema)

In [ ]:
for i in os.scandir(corpusDir):
    if i.name.endswith(".xml") and i.is_file():
        validate_corpus_file(i, corpusDir)

In [ ]:
def make_clickable(source, line=None):
    link = source.replace('../../','https://github.com/wibarab/corpus-data/blob/main/')
    if line:
        return f'<a href="{link}#L{line}" target="_blank">{source}</a>'
    else:
        return f'<a href="{link}" target="_blank">{source}</a>'

In [ ]:
validation_html_begin = """<!doctype html>
<html>
<head>
    <meta charset=\"utf-8\" />
    <title>WIBARAB corpus validation report</title>
    <style>
    body {
        font-family: Arial, sans-serif;
        font-size: 13px;
        margin: 20px;
    }
    table {
        font-family: Arial, sans-serif;
        font-size: 13px;
        border-collapse: collapse;
        width: 100%;
    }
    th, td {
        border: 1px solid #ccc;
        padding: 6px;
        text-align: left;
        min-width: 50px;
        max-width: 300px;
        overflow-wrap: break-word;
        word-break: break-word;
    }
    thead {
        background: #eee;
    }
    #tableFilter {
        margin-bottom: 12px;
    }
    details.file-group {
        margin-bottom: 8px;
        border: 1px solid #ccc;
        border-radius: 4px;
    }
    details.file-group summary {
        background: #eee;
        padding: 8px 12px;
        cursor: pointer;
        font-weight: bold;
        list-style: none;
        user-select: none;
    }
    details.file-group summary::-webkit-details-marker { display: none; }
    details.file-group summary::before {
        content: "\\25B6  ";
        font-size: 10px;
    }
    details.file-group[open] summary::before {
        content: "\\25BC  ";
    }
    .file-errors {
        padding: 8px;
    }
    .error-count {
        font-weight: normal;
        color: #c00;
        margin-left: 8px;
    }
    .file-path {
        font-weight: normal;
        color: #666;
        margin-left: 8px;
        font-size: 11px;
    }
    #resultSummary {
        margin-left: 12px;
        color: #555;
        font-size: 12px;
    }
</style>
</head>
<body>
<h1>WIBARAB corpus validation report</h1>
"""
validation_filter_ui = """<label for=\"tableFilter\">Filter rows:</label>
<input id=\"tableFilter\" type=\"text\" placeholder=\"Type to filter rows or filenames\" onkeyup=\"filterValidationTable()\" style=\"margin-left:8px;width:320px\" />
<button onclick=\"document.querySelectorAll('details.file-group').forEach(d=>d.open=true)\" style=\"margin-left:8px\">Expand all</button>
<button onclick=\"document.querySelectorAll('details.file-group').forEach(d=>d.open=false)\" style=\"margin-left:4px\">Collapse all</button>
<span id=\"resultSummary\"></span>
<div style=\"margin-bottom:12px\"></div>
"""
validation_filter_script = """<script>
function updateSummary(visibleRows, totalRows, visibleFiles, totalFiles) {
    const el = document.getElementById("resultSummary");
    if (!el) return;
    if (visibleRows === totalRows) {
        el.textContent = totalRows + " error" + (totalRows !== 1 ? "s" : "") + " in " + totalFiles + " file" + (totalFiles !== 1 ? "s" : "");
    } else {
        el.textContent = visibleRows + " of " + totalRows + " error" + (totalRows !== 1 ? "s" : "") + " in " + visibleFiles + " of " + totalFiles + " file" + (totalFiles !== 1 ? "s" : "");
    }
}

function filterValidationTable() {
    const input = document.getElementById("tableFilter");
    const filter = input.value.toLowerCase();
    const groups = document.querySelectorAll("details.file-group");
    let totalRows = 0, visibleRows = 0, totalFiles = groups.length, visibleFiles = 0;

    groups.forEach(function(group) {
        const summaryTxt = group.querySelector("summary").textContent.toLowerCase();
        const rows = group.querySelectorAll("tbody tr");
        let anyRowVisible = false;
        totalRows += rows.length;

        rows.forEach(function(row) {
            const txt = row.textContent || row.innerText;
            const visible = !filter || txt.toLowerCase().indexOf(filter) > -1;
            row.style.display = visible ? "" : "none";
            if (visible) { anyRowVisible = true; visibleRows++; }
        });

        // if the filename itself matches, show all rows in this group
        if (filter && summaryTxt.indexOf(filter) > -1) {
            rows.forEach(function(row) {
                if (row.style.display === "none") { row.style.display = ""; visibleRows++; }
            });
            anyRowVisible = true;
        }

        group.style.display = (!filter || anyRowVisible) ? "" : "none";
        if (anyRowVisible) visibleFiles++;
        if (filter && anyRowVisible) group.open = true;
    });

    updateSummary(visibleRows, totalRows, visibleFiles, totalFiles);
}

window.addEventListener("DOMContentLoaded", function() {
    filterValidationTable();
});
</script>
"""
validation_html_end = """</body></html>"""


In [ ]:
def no_validation_error(out_file):
    with open(out_file, "w", encoding="utf-8") as f:
        f.write(validation_html_begin)
        f.write("<p>No validation errors found.</p>")
        f.write(validation_html_end)

In [ ]:
if len(validationErrors) > 0:
    df_err = pd.DataFrame(data=validationErrors)
    print(f"found {len(validationErrors)} validation errors across {df_err['source'].nunique()} file(s)")
    df_err["link"] = df_err.apply(
        lambda x: make_clickable(x["source"], x.get("line")), axis=1
    )
    for col in ("entry", "relativePath", "type"):
        if col in df_err.columns:
            df_err.drop(columns=[col], inplace=True)

    display_cols = [c for c in df_err.columns if c not in ("source",)]

    html_parts = []
    for source, group in df_err.groupby("source"):
        count = len(group)
        short_name = os.path.basename(source)
        html_parts.append('<details class="file-group">')
        html_parts.append(
            f'<summary>{short_name}'
            f'<span class="error-count">({count} error{"s" if count != 1 else ""})</span>'
            f'<span class="file-path">{source}</span></summary>'
        )
        html_parts.append('<div class="file-errors">')
        html_parts.append(
            group[display_cols].to_html(render_links=True, escape=False, index=False)
        )
        html_parts.append("</div></details>")

    with open(errorReport, "w", encoding="utf-8") as f:
        f.write(validation_html_begin)
        f.write(validation_filter_ui)
        f.write("\n".join(html_parts))
        f.write(validation_filter_script)
        f.write(validation_html_end)
else:
    no_validation_error(errorReport)
